# Finding Redundant 1D Subspaces in LLM Activations

This notebook discovers **redundant 1-dimensional subspaces** in language model activations: directions where:
1. Activations have **large projections** (model actively uses this direction)
2. **Removing** this projection has **minimal impact** on output token distributions

In [ ]:
import sys
import pathlib

# set pythonpath to the main module directory
module_dir = pathlib.Path("..").parent.resolve().parent
if str(module_dir) not in sys.path:
    sys.path.append(str(module_dir))

In [ ]:
from scripts.utils.load_dataset import DatasetName, load_dataset

ds = load_dataset(DatasetName.OASST2)

In [ ]:
d = ds[0]
d

In [ ]:
import torch
from src.utils.env import set_seed

set_seed(42)

# torch.set_float32_matmul_precision("high")

In [ ]:
import torch
from scripts.utils.load_model import load_model
from src.model import TargetedModel
from src.aliases import Conv

model_name = "meta-llama/Llama-2-7b-chat-hf"

print(f"====== Testing model {model_name} ======")
model, tokenizer = load_model(model_name)
targeted_model = TargetedModel(model, tokenizer, is_chat=True)
print(f"Model {model_name}")
print(model)

In [ ]:
print(model.dtype)

In [ ]:
from scripts.utils.load_dataset import load_dataset
from src.data import TableLoader

ds_train, ds_val, ds_test = load_dataset("hh-rlhf")

dl_train = TableLoader(ds_train, batch_size=16, shuffle=True)
dl_val = TableLoader(ds_val, batch_size=8, shuffle=True)
dl_test = TableLoader(ds_test, batch_size=8, shuffle=True)